In [9]:
!pip install python-dotenv


[notice] A new release of pip available: 22.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
!pip install mysql-connector-python

In [10]:
from mysql.connector import connect
from dotenv import load_dotenv
import os

load_dotenv()

get_db_connection = lambda: connect(
    host="localhost",
    port="3306",
    user="root",
    password=os.getenv("MYSQL_PASSWORD"),
    database="test",
    autocommit=True
)




In [12]:
with get_db_connection() as conn:
    with conn.cursor() as cur:
        cur.execute("""
            INSERT INTO users(username,password)
            Values
            ("Ben", "1234"),
            ("Yan", "5678")
        """)

# Parametrized Inserts


In [13]:
query = "INSERT INTO users (username,password) VALUES (%s,%s);"

users = [
    ("bob"," 786"),
    ("aris", "thg"),
    ("annie", "jjhg")
]

In [14]:
with get_db_connection() as conn:
    with conn.cursor() as cur:
        cur.executemany(query,users)

# Prepared Statements

In [15]:
query = "INSERT INTO users (username,password) VALUES (%s,%s);"

In [17]:
millions_of_users = [
    ("greg"," 786"),
    ("kristine", "thg"),
    ("gel", "jjhg")
]

In [18]:
conn = get_db_connection()


In [20]:
cur = conn.cursor(prepared=True)

In [21]:
type(cur)

mysql.connector.cursor_cext.CMySQLCursorPrepared

In [22]:
with get_db_connection() as conn:
    with conn.cursor(prepared=True):
        cur.executemany(query, millions_of_users)

# Fetching

In [23]:
conn = get_db_connection()


In [24]:
conn = get_db_connection()

In [25]:
cur = conn.cursor()



In [26]:
cur.execute("SELECT * FROM users;")

In [28]:
cur.fetchall()

[(1, 'Robin', '1234'),
 (2, 'Mariel', '5678'),
 (3, 'Ben', '1234'),
 (4, 'Yan', '5678'),
 (5, 'bob', ' 786'),
 (6, 'aris', 'thg'),
 (7, 'annie', 'jjhg'),
 (8, 'Ben', '1234'),
 (9, 'Yan', '5678'),
 (10, 'bob', ' 786'),
 (11, 'aris', 'thg'),
 (12, 'annie', 'jjhg'),
 (13, 'greg', ' 786'),
 (14, 'kristine', 'thg'),
 (15, 'gel', 'jjhg')]

In [30]:
with get_db_connection() as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT * FROM users LIMIT 1;")
        print(cur.fetchone())

(1, 'Robin', '1234')


# Buffered Cursors

In [31]:
with get_db_connection() as conn:
    with conn.cursor(buffered=True) as cur:
        cur.execute("SELECT * FROM users LIMIT 1;")
        print(cur.fetchone())

(1, 'Robin', '1234')


# DICT cursors

In [34]:
with get_db_connection() as conn:
    with conn.cursor(dictionary=True) as cur:
        cur.execute("SELECT * FROM users LIMIT 3;")
        print(cur.fetchall())

[{'id': 1, 'username': 'Robin', 'password': '1234'}, {'id': 2, 'username': 'Mariel', 'password': '5678'}, {'id': 3, 'username': 'Ben', 'password': '1234'}]


In [36]:
with get_db_connection() as conn:
    with conn.cursor(dictionary=True) as cur:
        cur.execute("SELECT * FROM users LIMIT 3;")

        for user in cur.fetchall():
            print(user["username"])
            print(user.get("password"))
            
     

Robin
1234
Mariel
5678
Ben
1234


# Named Tuple Cursor

In [37]:
from collections import namedtuple


In [38]:
user = namedtuple("user","id username password")

In [39]:
user1 = user(1,"John","secret")

In [40]:
# immutable


In [ ]:
with get_db_connection() as conn:
    with conn.cursor(named_tuple=True) as cur:
        cur.execute("SELECT * FROM users LIMIT 3;")

        for user in cur.fetchall():
            print(user.username)
            print(user.password)

# Multi With Autocommit


In [48]:
sql_script = """
drop table if exists posts;

drop table if exists users;

create table users (
    id int auto_increment primary key,
    username varchar(20) not null,
    password varchar(20) not null,
    email varchar(20) not null
);

create table posts (
    id int auto_increment primary key,
    title varchar(20) not null,
    body varchar(20) not null,
    user_id int not null,
    foreign key (user_id) references users(id) on delete cascade);


insert into users (username, password, email) values ('johndoe', 'secret', 'jds@gmail.com');
insert into posts (title, body, user_id) values ('post1', 'body1', 1);
insert into posts (title, body, user_id) values ('post2', 'body2', 1);
insert into posts (title, body, user_id) values ('post3', 'body3', 1);
insert into posts (title, body, user_id) values ('post4', 'body4', 1);
"""

In [44]:
get_db_connection = lambda autocommit=False: connect(
    host="localhost",
    port="3306",
    user="root",
    password=os.getenv("MYSQL_PASSWORD"),
    database="test",
    autocommit=autocommit
)